# Simulation: Weiße und schwarze Kugeln

In einer Urne liegen $w$ weiße und $s$ schwarze Kugeln. Alle
$N=w+s$ Kugeln werden nacheinander **ohne Zurücklegen** gezogen.
Es sei

$$
A_j=\{\text{An Position }j\text{ liegt eine weiße Kugel}\}.
$$

Für die erste Position ist unmittelbar klar:

$$
P(A_1)=\frac{w}{N}.
$$

Die Simulation richtet den Blick auf die weiterführende Frage:

> Unterscheiden sich die Positionen hinsichtlich der Wahrscheinlichkeit
> für eine weiße Kugel?

Sie kann eine Vermutung sichtbar machen. Die mathematische Begründung
durch die Gleichberechtigung der Positionen ersetzt sie nicht.

## 1. Bibliotheken

In [ ]:
from math import comb
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

## 2. Eingaben

Alle veränderbaren Größen stehen an einer Stelle. Mit `seed = 42` ist die Simulation reproduzierbar.

In [ ]:
# Zusammensetzung der Urne
w = 2
s = 3

# Umfang der Simulation
anzahl_simulationen = 20_000
anzahl_folgen_anzeigen = 15
seed = 42

# Optionaler Ausblick: Zahl weißer Kugeln unter den ersten m Ziehungen
m = 3

# Abbildungen zusätzlich als PNG und PDF speichern?
abbildungen_speichern = False
ausgabeordner = Path("abbildungen")

## 3. Simulation

Eine Zeile des Arrays `farbfolgen` beschreibt eine vollständige
Farbfolge. Dabei steht `1` für Weiß und `0` für Schwarz.

In [ ]:
if not isinstance(w, (int, np.integer)) or w < 1:
    raise ValueError("w muss eine positive ganze Zahl sein.")
if not isinstance(s, (int, np.integer)) or s < 1:
    raise ValueError("s muss eine positive ganze Zahl sein.")
if (
    not isinstance(anzahl_simulationen, (int, np.integer))
    or anzahl_simulationen < 1
):
    raise ValueError("anzahl_simulationen muss eine positive ganze Zahl sein.")

N = w + s
if not isinstance(m, (int, np.integer)) or not 1 <= m <= N:
    raise ValueError("m muss zwischen 1 und N liegen.")

grundfolge = np.concatenate((
    np.ones(w, dtype=np.int8),
    np.zeros(s, dtype=np.int8),
))

rng = np.random.default_rng(seed)
farbfolgen = np.vstack([
    rng.permutation(grundfolge)
    for _ in range(anzahl_simulationen)
])

p_weiss = w / N
relative_haeufigkeiten_weiss = farbfolgen.mean(axis=0)

# Monte-Carlo-Standardabweichung der relativen Häufigkeit
# an einer festen Position
monte_carlo_standardabweichung = np.sqrt(
    p_weiss * (1 - p_weiss) / anzahl_simulationen
)

print(f"Theoretische Wahrscheinlichkeit für Weiß: {p_weiss:.4f}")
print(
    f"Zahl der simulierten Farbfolgen: {anzahl_simulationen:,}"
    .replace(",", ".")
)
print(
    "Mittelwert der relativen Häufigkeiten über alle Positionen: "
    f"{relative_haeufigkeiten_weiss.mean():.4f}"
)

## 4. Zeilenweise lesen: vollständige Farbfolgen

Jede Zeile zeigt einen vollständigen Versuch. In jeder Zeile liegen
genau $w$ weiße und $s$ schwarze Kugeln.

In [ ]:
def speichere_abbildung(fig, dateiname):
    """Speichert eine Abbildung bei Bedarf als PNG und als PDF."""
    if abbildungen_speichern:
        ausgabeordner.mkdir(parents=True, exist_ok=True)
        fig.savefig(ausgabeordner / f"{dateiname}.png", dpi=300, bbox_inches="tight")
        fig.savefig(ausgabeordner / f"{dateiname}.pdf", bbox_inches="tight")


anzahl_gezeigt = min(anzahl_folgen_anzeigen, anzahl_simulationen)
folgen = farbfolgen[:anzahl_gezeigt]

fig_breite = max(6.0, 0.72 * N)
fig_hoehe = max(3.0, min(11.0, 0.38 * anzahl_gezeigt))
fig, ax = plt.subplots(figsize=(fig_breite, fig_hoehe))

x = np.tile(np.arange(1, N + 1), anzahl_gezeigt)
y = np.repeat(np.arange(1, anzahl_gezeigt + 1), N)
punktgrundfolge = np.where(folgen.ravel() == 1, "tab:red", "black")
punktgroesse = max(35, min(360, 1800 / N))

ax.scatter(
    x,
    y,
    s=punktgroesse,
    c=punktgrundfolge,
    edgecolors="black",
    linewidths=0.6,
)
ax.set_xticks(np.arange(1, N + 1))
ax.set_yticks([])
ax.set_xlabel("Zugposition")
ax.set_title(f"{anzahl_gezeigt} simulierte Farbfolgen")
ax.set_xlim(0.45, N + 0.55)
ax.set_ylim(anzahl_gezeigt + 0.7, 0.3)
ax.set_aspect("equal", adjustable="box")

speichere_abbildung(fig, "simulierte_farbfolgen")
plt.show()

## 5. Spaltenweise lesen: Weiß an einer festen Position

Nun wird dieselbe Datenmatrix spaltenweise gelesen. Für jede Position
$j$ wird die relative Häufigkeit $h(A_j)$ bestimmt, mit der dort eine
weiße Kugel liegt.

Die gestrichelte Linie zeigt den theoretischen Wert

$$p=\frac{w}{N}.$$

Bei $r$ unabhängig simulierten Farbfolgen besitzt $h(A_j)$ die
Monte-Carlo-Standardabweichung

$$\sigma_{\mathrm{MC}}
  =\sqrt{\frac{p(1-p)}{r}}.$$

Die beiden dünnen horizontalen Linien markieren
$p-1{,}96\sigma_{\mathrm{MC}}$ und
$p+1{,}96\sigma_{\mathrm{MC}}$. Der Bereich dazwischen ist für jede
einzelne Position ein typischer Schwankungsbereich von ungefähr
$95\,\%$; er ist keine simultane Aussage über alle Positionen. Die
y-Achse zeigt bewusst nur einen Ausschnitt um $p$, damit die
zufallsbedingten Abweichungen sichtbar werden.

In [ ]:
positionen = np.arange(1, N + 1)
untere_grenze = max(
    0.0, p_weiss - 1.96 * monte_carlo_standardabweichung
)
obere_grenze = min(
    1.0, p_weiss + 1.96 * monte_carlo_standardabweichung
)

fig, ax = plt.subplots(figsize=(max(7.0, 0.65 * N), 4.8))
ax.axhline(
    untere_grenze, color="0.65", linestyle=":", linewidth=1.0
)
ax.axhline(
    obere_grenze, color="0.65", linestyle=":", linewidth=1.0
)
ax.axhline(p_weiss, color="0.2", linestyle="--", linewidth=1.4)
ax.vlines(
    positionen,
    p_weiss,
    relative_haeufigkeiten_weiss,
    color="tab:blue",
    linewidth=2,
)
ax.scatter(
    positionen,
    relative_haeufigkeiten_weiss,
    color="tab:blue",
    s=48,
    zorder=3,
)

# Die y-Achse wird bewusst um p herum vergrößert, damit die
# Monte-Carlo-Schwankungen sichtbar werden.
groesste_abweichung = np.max(
    np.abs(relative_haeufigkeiten_weiss - p_weiss)
)
halbe_hoehe = max(
    0.015,
    1.35 * groesste_abweichung,
    2.7 * monte_carlo_standardabweichung,
)
ax.set_ylim(
    max(0.0, p_weiss - halbe_hoehe),
    min(1.0, p_weiss + halbe_hoehe),
)

ax.set_xticks(positionen)
ax.yaxis.set_major_formatter(
    mtick.PercentFormatter(xmax=1.0, decimals=1)
)
ax.set_xlabel("Position")
ax.set_ylabel(r"Relative Häufigkeit $h(A_j)$")
ax.set_title("Weiße Kugel an einer festen Position")
ax.grid(axis="y", alpha=0.25)

speichere_abbildung(fig, "relative_haeufigkeit_nach_position")
plt.show()

print("Position   Relative Häufigkeit h(A_j)")
for position, haeufigkeit in zip(
    positionen, relative_haeufigkeiten_weiss
):
    print(f"{position:>7d} {haeufigkeit:>27.4f}")

Bei jeder Simulation gilt bereits exakt

$$
\frac{h(A_1)+\dots+h(A_N)}{N}=\frac{w}{N},
$$

denn jede simulierte Farbfolge enthält genau $w$ weiße Kugeln. Die
einzelnen Werte $h(A_j)$ stabilisieren sich dagegen erst mit wachsender
Wiederholungszahl beim gemeinsamen theoretischen Wert. Die Simulation
legt daher die Vermutung nahe:

$$
P(A_j)
=P(\text{Kugel an Position }j\text{ ist weiß})
=\frac{w}{N}
\qquad\text{für jedes }j.
$$

Nun entsteht die mathematische Frage:
**Was hat eine Position, was eine andere nicht hat?**

## 6. Gleichberechtigung ist nicht Unabhängigkeit

Die gleiche Wahrscheinlichkeit für Weiß an allen Positionen bedeutet nicht, dass die Farben an den Positionen unabhängig sind. In jeder Zeile steht insgesamt genau $w$-mal Weiß. Ist an einer Position bereits Weiß aufgetreten, verändert sich die Zusammensetzung für die übrigen Positionen.

Die folgende Rechnung vergleicht die Simulation mit der exakten bedingten Wahrscheinlichkeit für die zweite Position.

In [ ]:
if N >= 2:
    nach_weiss = farbfolgen[farbfolgen[:, 0] == 1, 1]
    nach_schwarz = farbfolgen[farbfolgen[:, 0] == 0, 1]

    empirisch_nach_weiss = nach_weiss.mean()
    empirisch_nach_schwarz = nach_schwarz.mean()
    exakt_nach_weiss = (w - 1) / (N - 1)
    exakt_nach_schwarz = w / (N - 1)

    print("Wahrscheinlichkeit für Weiß an Position 2")
    print(f"nach Weiß an Position 1:     Simulation {empirisch_nach_weiss:.4f} | exakt {exakt_nach_weiss:.4f}")
    print(f"nach Schwarz an Position 1: Simulation {empirisch_nach_schwarz:.4f} | exakt {exakt_nach_schwarz:.4f}")

## 7. Ausblick: Eine neue Verteilung entsteht aus dem Problem

Die Zufallsgröße $X$ zählt die weißen Kugeln unter den ersten $m$ Ziehungen. Ihre Wahrscheinlichkeitsverteilung ergibt sich aus dem Urnenproblem:

$$
P(X=k)
=\frac{\binom{w}{k}\binom{s}{m-k}}
{\binom{w+s}{m}}.
$$

Die Grafik vergleicht die simulierten relativen Häufigkeiten mit diesen exakten Wahrscheinlichkeiten.

In [ ]:
anzahl_weiss_unter_ersten_m = farbfolgen[:, :m].sum(axis=1)
k_min = max(0, m - s)
k_max = min(w, m)
k_werte = np.arange(k_min, k_max + 1)

simulierte_wahrscheinlichkeiten = np.array([
    np.mean(anzahl_weiss_unter_ersten_m == k)
    for k in k_werte
])
exakte_wahrscheinlichkeiten = np.array([
    comb(w, int(k)) * comb(s, m - int(k)) / comb(N, m)
    for k in k_werte
])

fig, ax = plt.subplots(figsize=(7.0, 4.8))
ax.vlines(k_werte, 0, exakte_wahrscheinlichkeiten, color="tab:blue", linewidth=5, alpha=0.65)
ax.scatter(k_werte, exakte_wahrscheinlichkeiten, color="tab:blue", s=48, zorder=3)
ax.scatter(k_werte, simulierte_wahrscheinlichkeiten, color="tab:red", marker="x", s=75, linewidths=2, zorder=4)

ax.set_xticks(k_werte)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0, decimals=0))
ax.set_xlabel(f"Zahl weißer Kugeln unter den ersten {m} Ziehungen")
ax.set_ylabel("Wahrscheinlichkeit / relative Häufigkeit")
ax.set_title("Hypergeometrische Verteilung und Simulation")
ax.set_ylim(0, min(1.0, 1.15 * max(exakte_wahrscheinlichkeiten.max(), simulierte_wahrscheinlichkeiten.max())))
ax.grid(axis="y", alpha=0.25)

speichere_abbildung(fig, "hypergeometrische_verteilung")
plt.show()

print("Blau: exakte Wahrscheinlichkeit | Rotes Kreuz: Simulation")

Der Erwartungswert allein wäre auch hier
nur die halbe Wahrheit. Für die hypergeometrisch verteilte Zufallsgröße
$X$ gilt mit $p=w/N$

$$
E(X)=m\,p,
\qquad
\operatorname{Var}(X)
=m\,p(1-p)\frac{N-m}{N-1}.
$$

Der letzte Faktor ist kleiner als $1$. Die Streuung ist daher kleiner
als bei $m$ unabhängigen Bernoulli-Versuchen mit derselben
Trefferwahrscheinlichkeit. Auch darin wird sichtbar:
**Gleichberechtigung ist nicht Unabhängigkeit.**

In [ ]:
erwartungswert_exakt = m * p_weiss
varianz_exakt = m * p_weiss * (1 - p_weiss) * (N - m) / (N - 1)

erwartungswert_simuliert = anzahl_weiss_unter_ersten_m.mean()
varianz_simuliert = anzahl_weiss_unter_ersten_m.var(ddof=0)

print("                         Simulation      exakt")
print(f"Erwartungswert          {erwartungswert_simuliert:10.4f} {erwartungswert_exakt:10.4f}")
print(f"Varianz                 {varianz_simuliert:10.4f} {varianz_exakt:10.4f}")
print(f"Standardabweichung      {np.sqrt(varianz_simuliert):10.4f} {np.sqrt(varianz_exakt):10.4f}")

## Mögliche Fragen an die Simulation

- Was bleibt bei jeder neuen Simulation exakt gleich?
- Was stabilisiert sich erst bei vielen Wiederholungen?
- Was ist beim zeilenweisen, was beim spaltenweisen Lesen zu erkennen?
- Welche Vermutung legt die Grafik nahe -- und was kann sie nicht
  beweisen?
- Woran lässt sich erkennen, dass Gleichberechtigung der Positionen
  keine Unabhängigkeit bedeutet?
- Was bedeutet die Gleichberechtigung, wenn nur ein einziges Mal
  zwischen der ersten und der letzten Position gewählt wird?
- Welche neue Fragestellung führt von den einzelnen Positionen zur
  hypergeometrischen Verteilung?